In [3]:
import os
import glob
import numpy as np
import nibabel as nib
import SimpleITK as sitk
from tqdm import tqdm
import sklearn # Required for train_test_split, ensure it's installed if not already

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from monai.transforms import (
    Compose, Orientationd, Spacingd, CropForegroundd,
    ScaleIntensityRanged, NormalizeIntensityd, ToTensord,
    RandSpatialCropd, RandFlipd, RandGaussianNoised, RandAdjustContrastd,
    RandBiasFieldd, EnsureChannelFirstd
)
from monai.networks.nets import UNet, SegResNet
from monai.data import decollate_batch
from monai.metrics import DiceMetric
from monai.losses import DiceLoss, DiceCELoss

# Conditional import for intensity_normalization
_HAS_INTENSITY_NORMALIZATION = False
try:
    # Attempt import directly from top-level or other common sub-modules if 'normalize' doesn't work
    # Or, specifically target a version if the API changed.
    # The common path is intensity_normalization.normalize
    from intensity_normalization.normalize import nyul_train_standard_scale, nyul_apply_standard_scale
    _HAS_INTENSITY_NORMALIZATION = True
except ImportError as e:
    print(f"Warning: 'intensity-normalization.normalize' module or its specific functions (nyul_train_standard_scale, nyul_apply_standard_scale) not importable: {e}")
    print("Nyul & Udupa normalization will be skipped. Please ensure 'intensity-normalization' is properly installed (e.g., version 1.7.0) and its API matches.")

# Conditional import for itk-elastix (still challenging to install via pip)
_HAS_ITK_ELASTIX = False
try:
    import itk # itk-elastix is often used with itk directly
    _HAS_ITK_ELASTIX = True
except ImportError:
    print("Warning: 'itk-elastix' not directly importable. Robust registration features will be limited.")


# --- Offline Preprocessing Functions (Conceptual / External) ---
# These functions illustrate steps that *should* ideally be run once offline
# to prepare perfectly aligned and normalized data.
# Due to SimpleITK-Elastix installation challenges, these are provided as a conceptual
# guide for manual execution outside this notebook's direct runtime.

def apply_n4_bias_correction(image_path, output_path):
    """
    Applies N4 Bias Field Correction to a NIfTI image.
    NOTE: This is a conceptual function. You should run this offline.
    """
    try:
        input_image = sitk.ReadImage(image_path, sitk.sitkFloat32)
        mask_image = sitk.OtsuThreshold(input_image, 0, 1, 200)

        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrector.SetMaximumNumberOfIterations([100, 100, 60, 40])
        corrected_image_sitk = corrector.Execute(input_image, mask_image)
        log_bias_field = corrector.GetLogBiasFieldAsImage(input_image)
        corrected_image_full_resolution = input_image / sitk.Exp(log_bias_field)

        sitk.WriteImage(corrected_image_full_resolution, output_path)
        print(f"N4 corrected and saved: {output_path}")
    except Exception as e:
        print(f"Error applying N4 to {image_path}: {e}")

def apply_nyul_normalization(image_path, output_path, standard_scale):
    """
    Applies Nyul & Udupa normalization to a NIfTI image.
    NOTE: This is a conceptual function. You should run this offline.
    Requires 'intensity-normalization' library.
    """
    if not _HAS_INTENSITY_NORMALIZATION:
        print("Skipping Nyul normalization: 'intensity-normalization' not available or functions not found.")
        return
    try:
        img = nib.load(image_path)
        data = img.get_fdata()
        normalized_data = nyul_apply_standard_scale(data, standard_scale)
        normalized_img = nib.Nifti1Image(normalized_data.astype(np.float32), img.affine, img.header)
        nib.save(normalized_img, output_path)
        print(f"Nyul normalized and saved: {output_path}")
    except Exception as e:
        print(f"Error applying Nyul normalization to {image_path}: {e}")

def register_image_and_mask(fixed_image_path, moving_image_path, moving_mask_path, output_dir, subject_id, modality_name):
    """
    Registers a moving image to a fixed image using SimpleITK.Elastix (if available)
    and applies the same transformation to the corresponding mask.
    NOTE: This is a conceptual function. It's recommended to run robust registration offline.
    """
    if not _HAS_ITK_ELASTIX:
        print(f"Skipping Elastix registration for {modality_name}: 'itk-elastix' not available.")
        return

    try:
        fixed_image = sitk.ReadImage(fixed_image_path, sitk.sitkFloat32)
        moving_image = sitk.ReadImage(moving_image_path, sitk.sitkFloat32)
        moving_mask = sitk.ReadImage(moving_mask_path, sitk.sitkUInt8)

        parameter_object = sitk.ParameterObject()
        parameter_object.AddParameterMap(parameter_object.GetDefaultParameterMap("rigid"))
        parameter_object.AddParameterMap(parameter_object.GetDefaultParameterMap("affine"))
        parameter_object.AddParameterMap(parameter_object.GetDefaultParameterMap("bspline"))

        for i in range(parameter_object.GetNumberOfParameterMaps()):
            parameter_object.SetParameter(i, "Metric", "AdvancedMattesMutualInformation")

        elastix_filter = sitk.ElastixImageFilter()
        elastix_filter.SetFixedImage(fixed_image)
        elastix_filter.SetMovingImage(moving_image)
        elastix_filter.SetParameterObject(parameter_object)
        elastix_filter.Execute()

        registered_image = elastix_filter.GetResultImage()
        sitk.WriteImage(registered_image, os.path.join(output_dir, f'{subject_id}_{modality_name}_preprocessed.nii.gz'))

        transform_param_map = elastix_filter.GetTransformParameterMap()
        transformix_filter = sitk.TransformixImageFilter()
        transformix_filter.SetMovingImage(moving_mask)
        transformix_filter.SetTransformParameterMap(transform_param_map)
        transformix_filter.Execute()

        registered_mask = transformix_filter.GetResultImage()
        sitk.WriteImage(registered_mask, os.path.join(output_dir, f'{subject_id}_lesion-msk_registered.nii.gz'))

        print(f"Registered {modality_name} and its mask for {subject_id}")
    except Exception as e:
        print(f"Error during registration for {subject_id} {modality_name}: {e}")

# --- End Offline Preprocessing Functions ---

# --- Modified ISLESDataset3D to load RAW files and apply minimal on-the-fly standardization ---
# This version loads raw files and relies on MONAI transforms for initial spatial alignment
# and intensity standardization. For full "perfect alignment," run Elastix offline.
class ISLESDataset3D(Dataset):
    def __init__(self, raw_data_root_dir, raw_mask_root_dir):
        self.samples = []
        print(f"Scanning for 3D samples in raw data root: {raw_data_root_dir}")

        subject_dirs = sorted(glob.glob(os.path.join(raw_data_root_dir, 'sub-*')))

        for subject_dir in subject_dirs:
            subject_id = os.path.basename(subject_dir)
            ses_dwi_dir = os.path.join(subject_dir, "ses-0001", "dwi")
            ses_anat_dir = os.path.join(subject_dir, "ses-0001", "anat")
            mask_dir = os.path.join(raw_mask_root_dir, subject_id, "ses-0001")

            dwi_files = glob.glob(os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_dwi.nii.gz'))
            adc_files = glob.glob(os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_adc.nii.gz'))
            flair_files = glob.glob(os.path.join(ses_anat_dir, f'{subject_id}_ses-0001_FLAIR.nii.gz'))
            mask_files = glob.glob(os.path.join(mask_dir, f'{subject_id}_ses-0001_msk.nii.gz'))

            if dwi_files and adc_files and flair_files and mask_files:
                self.samples.append({
                    "image_dwi": dwi_files[0],
                    "image_adc": adc_files[0],
                    "image_flair": flair_files[0],
                    "label": mask_files[0],
                    "subject_id": subject_id
                })
            else:
                print(f"Skipping subject {subject_id} due to missing files:")
                if not dwi_files: print(f"  Missing DWI: {os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_dwi.nii.gz')}")
                if not adc_files: print(f"  Missing ADC: {os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_adc.nii.gz')}")
                if not flair_files: print(f"  Missing FLAIR: {os.path.join(ses_anat_dir, f'{subject_id}_ses-0001_FLAIR.nii.')}")
                if not mask_files: print(f"  Missing Mask: {os.path.join(mask_dir, f'{subject_id}_ses-0001_lesion-msk.nii')}")

        print(f"Total 3D samples found: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample_paths = self.samples[idx]

        try:
            # Load images as SimpleITK objects, then convert to NumPy arrays.
            # MONAI transforms will handle further spatial/intensity standardization.
            dwi_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["image_dwi"], sitk.sitkFloat32)).astype(np.float32)
            adc_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["image_adc"], sitk.sitkFloat32)).astype(np.float32)
            flair_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["image_flair"], sitk.sitkFloat32)).astype(np.float32)
            mask_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["label"], sitk.sitkUInt8)).astype(np.float32)

            mask_data = (mask_data > 0.5).astype(np.float32)

            image_stacked = np.stack([dwi_data, adc_data, flair_data], axis=0)

            return {"image": image_stacked, "label": mask_data, "subject_id": sample_paths["subject_id"]}

        except Exception as e:
            print(f"Error loading or processing sample {sample_paths.get('subject_id', 'N/A')}: {e}")
            raise

# --- Data Loading and Preprocessing (Step 1: Define Data Paths and Structure) ---

# Adjusting data_root based on your "refer previous directory ../data" instruction.
# This assumes your notebook is in a directory like `/content/project/notebooks/`
# and your ISLES-2022 data is located at `/content/project/data/ISLES-2022/`.

base_project_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
base_data_dir = os.path.join(base_project_dir, 'data')

data_root =base_data_dir
mask_root_for_raw = os.path.join(base_data_dir, 'derivatives')

# This `preprocessed_data_root` is where *you would save* the output of offline preprocessing.
# The `ISLESDataset3D` in this version is *not* reading from here; it's reading from `data_root` (raw data).
preprocessed_data_root = os.path.join(base_data_dir, 'ISLES-2022_preprocessed')

print(f"Base Project Directory: {base_project_dir}")
print(f"Resolved Raw Data Root: {data_root}")
print(f"Resolved Raw Mask Root: {mask_root_for_raw}")
print(f"Conceptual Preprocessed Data Root (for offline saving): {preprocessed_data_root}")

os.makedirs(preprocessed_data_root, exist_ok=True)

# --- Conceptual Offline Preprocessing Execution (You would run this once, externally) ---
# This block is for demonstration only. Do NOT run this unless you intend to
# execute the entire, lengthy preprocessing pipeline externally.
# It requires `intensity-normalization` and potentially `itk-elastix` to be functional.
#
# all_raw_subject_dirs = sorted(glob.glob(os.path.join(data_root, 'sub-*')))
#
# for raw_subject_dir in tqdm(all_raw_subject_dirs, desc="Conceptual Offline Preprocessing"):
#     raw_subject_id = os.path.basename(raw_subject_dir)
#     output_subject_dir = os.path.join(preprocessed_data_root, raw_subject_id, 'ses-0001')
#     os.makedirs(output_subject_dir, exist_ok=True)
#
#     current_raw_dwi_path = os.path.join(raw_subject_dir, 'ses-0001', 'dwi', f'{raw_subject_id}_ses-0001_dwi.nii.gz')
#     current_raw_adc_path = os.path.join(raw_subject_dir, 'ses-0001', 'dwi', f'{raw_subject_id}_ses-0001_adc.nii.gz')
#     current_raw_flair_path = os.path.join(raw_subject_dir, 'ses-0001', 'anat', f'{raw_subject_id}_ses-0001_FLAIR.nii.gz')
#     current_raw_mask_path = os.path.join(mask_root_for_raw, raw_subject_id, 'ses-0001', f'{raw_subject_id}_ses-0001_lesion-msk.nii.gz')
#
#     if not all(os.path.exists(f) for f in [current_raw_dwi_path, current_raw_adc_path, current_raw_flair_path, current_raw_mask_path]):
#         # print(f"Skipping conceptual preprocessing for {raw_subject_id}: Raw files not found.") # Uncomment for verbose skipping
#         continue
#
#     # N4 Bias Correction
#     n4_dwi_path = os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_n4.nii.gz')
#     n4_adc_path = os.path.join(output_subject_dir, f'{raw_subject_id}_adc_n4.nii.gz')
#     n4_flair_path = os.path.join(output_subject_dir, f'{raw_subject_id}_flair_n4.nii.gz')
#     # apply_n4_bias_correction(current_raw_dwi_path, n4_dwi_path)
#     # apply_n4_bias_correction(current_raw_adc_path, n4_adc_path)
#     # apply_n4_bias_correction(current_raw_flair_path, n4_flair_path)
#
#     # Nyul & Udupa Normalization (Requires pre-trained scales and _HAS_INTENSITY_NORMALIZATION to be True)
#     # This part assumes you have already run nyul_train_standard_scale on a representative subset
#     # of your N4-corrected training data and saved the scales.
#     # If you run Nyul, replace n4_paths with nyul_paths in registration.
#     #
#     # try:
#     #     dwi_standard_scale = np.load(os.path.join(preprocessed_data_root, 'dwi_nyul_scale.npy'))
#     #     adc_standard_scale = np.load(os.path.join(preprocessed_data_root, 'adc_nyul_scale.npy'))
#     #     flair_standard_scale = np.load(os.path.join(preprocessed_data_root, 'flair_nyul_scale.npy'))
#     #
#     #     nyul_dwi_path = os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_n4_nyul.nii.gz')
#     #     nyul_adc_path = os.path.join(output_subject_dir, f'{raw_subject_id}_adc_n4_nyul.nii.gz')
#     #     nyul_flair_path = os.path.join(output_subject_dir, f'{raw_subject_id}_flair_n4_nyul.nii.gz')
#     #
#     #     apply_nyul_normalization(n4_dwi_path, nyul_dwi_path, dwi_standard_scale)
#     #     apply_nyul_normalization(n4_adc_path, nyul_adc_path, adc_standard_scale)
#     #     apply_nyul_normalization(n4_flair_path, nyul_flair_path, flair_standard_scale)
#     # except FileNotFoundError:
#     #     # print(f"Skipping Nyul normalization for {raw_subject_id}: Standard scales not found.") # Uncomment for verbose skipping
#     #     pass # Skip if scales are not pre-trained
#
#     # Multimodal Registration (Elastix)
#     # fixed_image_for_reg = os.path.join(output_subject_dir, f'{raw_subject_id}_flair_n4_nyul.nii.gz') # Or _n4.nii.gz if Nyul skipped
#     # moving_dwi_for_reg = os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_n4_nyul.nii.gz') # Or _n4.nii.gz
#     # moving_adc_for_reg = os.path.join(output_subject_dir, f'{raw_subject_id}_adc_n4_nyul.nii.gz') # Or _n4.nii.gz
#     #
#     # register_image_and_mask(fixed_image_for_reg, moving_dwi_for_reg, current_raw_mask_path, output_subject_dir, raw_subject_id, 'dwi')
#     # register_image_and_mask(fixed_image_for_reg, moving_adc_for_reg, current_raw_mask_path, output_subject_dir, raw_subject_id, 'adc')
#
#     # Copy/rename FLAIR as "preprocessed" as it's the fixed reference
#     # shutil.copy(fixed_image_for_reg, os.path.join(output_subject_dir, f'{raw_subject_id}_flair_preprocessed.nii.gz'))
#
# print("--- Conceptual offline preprocessing section. For production, run these steps externally. ---")


# Create full dataset instance using the RAW data root (as Elastix is problematic to install)
# This means MONAI transforms will handle initial spatial standardization (Orientation, Spacing).
# For perfect alignment, you *must* run the full N4+Nyul+Elastix pipeline offline and then
# modify ISLESDataset3D to read from `preprocessed_data_root`.
full_dataset = ISLESDataset3D(data_root, mask_root_for_raw)

# Split indices for training and validation
from sklearn.model_selection import train_test_split
train_indices, val_indices = train_test_split(range(len(full_dataset)), test_size=0.2, random_state=42)

# Create subset datasets
train_ds = torch.utils.data.Subset(full_dataset, train_indices)
val_ds = torch.utils.data.Subset(full_dataset, val_indices)

print(f"Total training subjects: {len(train_ds)}")
print(f"Total validation subjects: {len(val_ds)}")

# --- MONAI Transforms for Data Augmentation and Spatial Standardization ---
# These transforms will be applied to the NumPy arrays returned by ISLESDataset3D.__getitem__
# They handle channel addition, orientation, spacing, and intensity transforms.

target_spacing = (1.0, 1.0, 1.0) # Example target spacing for final data
roi_size = (128, 128, 128) # Example patch size for training

train_transforms = Compose(
    [
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=target_spacing, mode=("bilinear", "nearest")),
        CropForegroundd(keys=["image", "label"], source_key="image", k_divisible=roi_size),
        ScaleIntensityRanged(keys="image", a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys="image", subtrahend=0.5, divisor=0.5),

        RandSpatialCropd(keys=["image", "label"], roi_size=roi_size, random_size=False, random_center=True),
        RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=1, prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=2, prob=0.5),
        RandGaussianNoised(keys=["image"], prob=0.1, std=0.01),
        RandAdjustContrastd(keys=["image"], prob=0.1, gamma=(0.7, 1.3)),
        RandBiasFieldd(keys=["image"], prob=0.1, coeff_range=(0.0, 0.05)),

        ToTensord(keys=["image", "label"]),
    ]
)

val_transforms = Compose(
    [
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=target_spacing, mode=("bilinear", "nearest")),
        CropForegroundd(keys=["image", "label"], source_key="image", k_divisible=roi_size),
        ScaleIntensityRanged(keys="image", a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys=["image"], subtrahend=0.5, divisor=0.5),
        ToTensord(keys=["image", "label"]),
    ]
)

# Apply MONAI transforms to the subset datasets
train_ds.transform = train_transforms
val_ds.transform = val_transforms

# Create MONAI DataLoaders
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=4, pin_memory=torch.cuda.is_available())

Nyul & Udupa normalization will be skipped. Please ensure 'intensity-normalization' is properly installed (e.g., version 1.7.0) and its API matches.
Base Project Directory: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set
Resolved Raw Data Root: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data
Resolved Raw Mask Root: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\derivatives
Conceptual Preprocessed Data Root (for offline saving): c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\ISLES-2022_preprocessed
Scanning for 3D samples in raw data root: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data
Skipping subject sub-strokecase0204 due to missing files:
  Missing DWI: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\sub-strokecase0204\ses-0001\dwi\sub-strokecase0204_ses-0001_dwi.nii.gz
  Missing ADC: c:\Users\punit\OneDrive

In [4]:
# Required imports
import os
import glob
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import matplotlib.pyplot as plt
from intensity_normalization.normalize import nyul
from intensity_normalization.plot import histogram
from tqdm import tqdm
import shutil

def create_comprehensive_overlay(fixed_path, moving_path, registered_path, mask_path, output_dir, subject_id):
    """Create a comprehensive visualization showing all three modalities with mask overlay."""
    try:
        # Load images and mask
        fixed = sitk.ReadImage(fixed_path)
        moving = sitk.ReadImage(moving_path)
        registered = sitk.ReadImage(registered_path)
        mask = sitk.ReadImage(mask_path)
        
        # Convert to numpy arrays
        fixed_array = sitk.GetArrayFromImage(fixed)
        moving_array = sitk.GetArrayFromImage(moving)
        registered_array = sitk.GetArrayFromImage(registered)
        mask_array = sitk.GetArrayFromImage(mask)
        
        # Get middle slice for each image - make sure it's within bounds
        mid_slice_fixed = min(fixed_array.shape[2] // 2, fixed_array.shape[2] - 1)
        mid_slice_moving = min(moving_array.shape[2] // 2, moving_array.shape[2] - 1)
        mid_slice_registered = min(registered_array.shape[2] // 2, registered_array.shape[2] - 1)
        mid_slice_mask = min(mask_array.shape[2] // 2, mask_array.shape[2] - 1)
        
        # Create figure with all three images in one row
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # FLAIR (Fixed) with mask
        axes[0].imshow(fixed_array[:, mid_slice_fixed, :], cmap='gray')
        axes[0].imshow(mask_array[:, mid_slice_mask, :], cmap='Greens', alpha=0.3)
        axes[0].set_title('FLAIR (Fixed) with Mask')
        axes[0].axis('off')
        
        # Moving (DWI/ADC) with mask
        axes[1].imshow(moving_array[:, mid_slice_moving, :], cmap='gray')
        axes[1].imshow(mask_array[:, mid_slice_mask, :], cmap='Greens', alpha=0.3)
        axes[1].set_title('Moving Image with Mask')
        axes[1].axis('off')
        
        # Registered with mask
        axes[2].imshow(registered_array[:, mid_slice_registered, :], cmap='gray')
        axes[2].imshow(mask_array[:, mid_slice_mask, :], cmap='Greens', alpha=0.3)
        axes[2].set_title('Registered Image with Mask')
        axes[2].axis('off')
        
        # Add color legend for mask
        cax = fig.add_axes([0.92, 0.2, 0.02, 0.6])
        plt.colorbar(plt.cm.ScalarMappable(cmap='Greens'), cax=cax)
        cax.set_title('Mask', pad=10)
        
        plt.tight_layout()
        plt.subplots_adjust(right=0.9)
        
        # Save the comprehensive visualization
        output_path = os.path.join(output_dir, f'{subject_id}_all_modalities_overlay.png')
        plt.savefig(output_path, bbox_inches='tight')
        plt.close()
        
        logger.info(f"Saved comprehensive overlay visualization to: {output_path}")
        
        # Also save separate views for better detail
        # FLAIR view
        plt.figure(figsize=(6, 6))
        plt.imshow(fixed_array[:, mid_slice_fixed, :], cmap='gray')
        plt.imshow(mask_array[:, mid_slice_mask, :], cmap='Greens', alpha=0.3)
        plt.title('FLAIR with Mask')
        plt.axis('off')
        plt.savefig(os.path.join(output_dir, f'{subject_id}_flair_overlay.png'), bbox_inches='tight')
        plt.close()
        
        # Moving view
        plt.figure(figsize=(6, 6))
        plt.imshow(moving_array[:, mid_slice_moving, :], cmap='gray')
        plt.imshow(mask_array[:, mid_slice_mask, :], cmap='Greens', alpha=0.3)
        plt.title('Moving Image with Mask')
        plt.axis('off')
        plt.savefig(os.path.join(output_dir, f'{subject_id}_moving_overlay.png'), bbox_inches='tight')
        plt.close()
        
        # Registered view
        plt.figure(figsize=(6, 6))
        plt.imshow(registered_array[:, mid_slice_registered, :], cmap='gray')
        plt.imshow(mask_array[:, mid_slice_mask, :], cmap='Greens', alpha=0.3)
        plt.title('Registered Image with Mask')
        plt.axis('off')
        plt.savefig(os.path.join(output_dir, f'{subject_id}_registered_overlay.png'), bbox_inches='tight')
        plt.close()
        
        logger.info(f"Saved separate overlay views for {subject_id}")
        
    except Exception as e:
        logger.error(f"Error creating comprehensive overlay visualization: {str(e)}")
# Preprocessing functions
def apply_n4_bias_correction(input_path, output_path):
    """Apply N4 bias field correction to an image."""
    image = sitk.ReadImage(input_path)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrected_image = corrector.Execute(image)
    sitk.WriteImage(corrected_image, output_path)

def apply_nyul_normalization(input_path, output_path, standard_scale):
    """Apply Nyul & Udupa normalization using pre-trained scales."""
    img = nib.load(input_path)
    img_data = img.get_fdata()
    
    normalizer = nyul.NyulNormalizer()
    normalizer.fit(img_data)
    normalized_data = normalizer.transform(img_data, standard_scale)
    
    normalized_img = nib.Nifti1Image(normalized_data, img.affine, img.header)
    nib.save(normalized_img, output_path)

def preprocess_image(image_path, target_spacing=(2.0, 2.0, 2.0)):
    """Preprocess image to a standard resolution and size."""
    # Read image
    image = sitk.ReadImage(image_path)
    
    # Get original spacing and size
    original_spacing = np.array(image.GetSpacing())
    original_size = np.array(image.GetSize())
    
    # Calculate new size based on target spacing
    new_size = np.round(original_size * (original_spacing / target_spacing)).astype(int)
    
    # Create resampling filter
    resample = sitk.ResampleImageFilter()
    resample.SetOutputSpacing(target_spacing)
    resample.SetSize(new_size.tolist())
    resample.SetOutputDirection(image.GetDirection())
    resample.SetOutputOrigin(image.GetOrigin())
    resample.SetTransform(sitk.Transform())
    resample.SetDefaultPixelValue(0)
    
    # Use appropriate interpolator
    if image.GetPixelID() in [sitk.sitkUInt8, sitk.sitkInt8, sitk.sitkUInt16, sitk.sitkInt16]:
        resample.SetInterpolator(sitk.sitkNearestNeighbor)
    else:
        resample.SetInterpolator(sitk.sitkLinear)
    
    # Resample the image
    resampled_image = resample.Execute(image)
    
    # Print debug info
    print(f"Original size: {original_size}, Original spacing: {original_spacing}")
    print(f"New size: {resampled_image.GetSize()}, New spacing: {resampled_image.GetSpacing()}")
    
    return resampled_image

# Required imports
import os
import glob
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import matplotlib.pyplot as plt
from intensity_normalization.normalize import nyul
from intensity_normalization.plot import histogram
from tqdm import tqdm
import shutil
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def preprocess_image(image_path, target_spacing=(2.0, 2.0, 2.0)):
    """Preprocess image to a standard resolution and size."""
    try:
        # Read image
        image = sitk.ReadImage(image_path)
        
        # Get original spacing and size
        original_spacing = np.array(image.GetSpacing())
        original_size = np.array(image.GetSize())
        
        # Calculate new size based on target spacing
        new_size = np.round(original_size * (original_spacing / target_spacing)).astype(int)
        
        # Create resampling filter
        resample = sitk.ResampleImageFilter()
        resample.SetOutputSpacing(target_spacing)
        resample.SetSize(new_size.tolist())
        resample.SetOutputDirection(image.GetDirection())
        resample.SetOutputOrigin(image.GetOrigin())
        resample.SetTransform(sitk.Transform())
        resample.SetDefaultPixelValue(0)
        
        # Use appropriate interpolator
        if image.GetPixelID() in [sitk.sitkUInt8, sitk.sitkInt8, sitk.sitkUInt16, sitk.sitkInt16]:
            resample.SetInterpolator(sitk.sitkNearestNeighbor)
        else:
            resample.SetInterpolator(sitk.sitkLinear)
        
        # Resample the image
        resampled_image = resample.Execute(image)
        
        # Log debug info
        logger.info(f"Processed {os.path.basename(image_path)}")
        logger.info(f"Original size: {original_size}, Original spacing: {original_spacing}")
        logger.info(f"New size: {resampled_image.GetSize()}, New spacing: {resampled_image.GetSpacing()}")
        
        return resampled_image
    except Exception as e:
        logger.error(f"Error preprocessing image {image_path}: {str(e)}")
        raise

def register_image_and_mask(fixed_image_path, moving_image_path, mask_path, output_dir, subject_id, modality):
    try:
        # Load images
        fixed = sitk.ReadImage(fixed_image_path)
        moving = sitk.ReadImage(moving_image_path)
        mask = sitk.ReadImage(mask_path)
        
        # Convert to float if not already
        fixed = sitk.Cast(fixed, sitk.sitkFloat32)
        moving = sitk.Cast(moving, sitk.sitkFloat32)
        
        # Create registration method
        registration_method = sitk.ImageRegistrationMethod()
        
        # Similarity metric settings
        registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
        registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
        registration_method.SetMetricSamplingPercentage(0.01)
        
        # Optimizer settings
        registration_method.SetOptimizerAsGradientDescent(
            learningRate=1.0,
            numberOfIterations=200,
            convergenceMinimumValue=1e-6,
            convergenceWindowSize=10
        )
        
        # Interpolator
        registration_method.SetInterpolator(sitk.sitkLinear)
        
        # Multi-resolution framework
        registration_method.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
        registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
        registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
        
        # Initial alignment - center the images
        initial_transform = sitk.CenteredTransformInitializer(
            fixed, moving, sitk.AffineTransform(fixed.GetDimension()),
            sitk.CenteredTransformInitializerFilter.GEOMETRY
        )
        
        registration_method.SetInitialTransform(initial_transform)
        
        try:
            # Execute registration
            final_transform = registration_method.Execute(fixed, moving)
            
            # Apply transform to moving image
            registered_image = sitk.Resample(
                moving, fixed, final_transform,
                sitk.sitkLinear, 0.0, moving.GetPixelID()
            )
            
            # Apply same transform to mask
            transformed_mask = sitk.Resample(
                mask, fixed, final_transform,
                sitk.sitkNearestNeighbor, 0.0, mask.GetPixelID()
            )
            
            # Save results
            registered_image_path = os.path.join(output_dir, f'{subject_id}_{modality}_registered.nii.gz')
            transformed_mask_path = os.path.join(output_dir, f'{subject_id}_ses-0001_lesion-msk_registered.nii.gz')
            
            sitk.WriteImage(registered_image, registered_image_path)
            sitk.WriteImage(transformed_mask, transformed_mask_path)
            
            print(f"Successfully registered {modality} for subject {subject_id}")
            
        except RuntimeError as e:
            print(f"Registration failed for {subject_id}, trying simpler approach: {str(e)}")
            
            # Use identity transform as fallback
            identity_transform = sitk.AffineTransform(fixed.GetDimension())
            identity_transform.SetIdentity()
            
            # Apply identity transform
            registered_image = sitk.Resample(
                moving, fixed, identity_transform,
                sitk.sitkLinear, 0.0, moving.GetPixelID()
            )
            
            transformed_mask = sitk.Resample(
                mask, fixed, identity_transform,
                sitk.sitkNearestNeighbor, 0.0, mask.GetPixelID()
            )
            
            # Save results
            registered_image_path = os.path.join(output_dir, f'{subject_id}_{modality}_registered_simple.nii.gz')
            transformed_mask_path = os.path.join(output_dir, f'{subject_id}_ses-0001_lesion-msk_registered_simple.nii.gz')
            
            sitk.WriteImage(registered_image, registered_image_path)
            sitk.WriteImage(transformed_mask, transformed_mask_path)
            
            print(f"Saved simple registration results for {subject_id}")
            
    except Exception as e:
        print(f"Error during registration for {subject_id}: {str(e)}")
        raise

def apply_n4_bias_correction(input_path, output_path):
    try:
        # Read input image with proper data type
        image = sitk.ReadImage(input_path, sitk.sitkFloat32)
        
        # Create a mask using Otsu thresholding
        mask_image = sitk.OtsuThreshold(image, 0, 1, 200)
        
        # Create N4 bias field correction filter
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrector.SetMaximumNumberOfIterations([100, 100, 60, 40])
        
        # Apply N4 bias correction
        corrected_image = corrector.Execute(image, mask_image)
        
        # Convert back to original data type if needed
        original_image = sitk.ReadImage(input_path)
        if original_image.GetPixelID() != sitk.sitkFloat32:
            corrected_image = sitk.Cast(corrected_image, original_image.GetPixelID())
        
        # Copy metadata from original image
        corrected_image.CopyInformation(original_image)
        
        # Write output
        sitk.WriteImage(corrected_image, output_path)
        print(f"Applied N4 bias correction to {output_path}")
        
    except Exception as e:
        print(f"Error applying N4 to {input_path}: {e}")
        raise
    
def apply_nyul_normalization(input_path, output_path, standard_scale):
    """Apply Nyul & Udupa normalization using pre-trained scales."""
    try:
        img = nib.load(input_path)
        img_data = img.get_fdata()
        
        normalizer = nyul.NyulNormalizer()
        normalizer.fit(img_data)
        normalized_data = normalizer.transform(img_data, standard_scale)
        
        normalized_img = nib.Nifti1Image(normalized_data, img.affine, img.header)
        nib.save(normalized_img, output_path)
        logger.info(f"Applied Nyul normalization to {output_path}")
    except Exception as e:
        logger.error(f"Error applying Nyul normalization: {str(e)}")
        raise

def plot_intensity_histogram(image_path, output_path):
    """Plot intensity histogram for visualization."""
    try:
        img = nib.load(image_path)
        img_data = img.get_fdata()
        
        # Create plot
        plt.figure(figsize=(10, 6))
        plt.hist(img_data.flatten(), bins=100, log=True)
        plt.title(f'Intensity Histogram - {os.path.basename(image_path)}')
        plt.xlabel('Intensity')
        plt.ylabel('Frequency')
        
        # Save plot
        plt.savefig(output_path)
        plt.close()
        
        logger.info(f"Saved histogram to: {output_path}")
        
        # Also save a 2D slice visualization
        slice_path = output_path.replace('.png', '_slice.png')
        mid_slice = img_data.shape[2] // 2
        plt.figure(figsize=(8, 8))
        plt.imshow(img_data[:, :, mid_slice], cmap='gray')
        plt.title(f'Mid-slice - {os.path.basename(image_path)}')
        plt.axis('off')
        plt.savefig(slice_path)
        plt.close()
        
        logger.info(f"Saved slice visualization to: {slice_path}")
        
    except Exception as e:
        logger.error(f"Error generating visualization for {image_path}: {str(e)}")

def plot_intensity_histogram(image_path, mask_path, output_path):
    """Plot intensity histogram and overlay mask visualization."""
    try:
        # Load image and mask
        img = nib.load(image_path)
        img_data = img.get_fdata()
        
        # Create plot
        plt.figure(figsize=(10, 6))
        plt.hist(img_data.flatten(), bins=100, log=True)
        plt.title(f'Intensity Histogram - {os.path.basename(image_path)}')
        plt.xlabel('Intensity')
        plt.ylabel('Frequency')
        
        # Save plot
        plt.savefig(output_path)
        plt.close()
        
        # Save a 2D slice visualization with mask overlay
        slice_path = output_path.replace('.png', '_slice.png')
        
        # Get middle slice - make sure it's within bounds
        mid_slice = min(img_data.shape[2] // 2, img_data.shape[2] - 1)
        
        # Create figure with both sagittal and coronal views
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        
        # Load mask
        mask = nib.load(mask_path)
        mask_data = mask.get_fdata()
        
        # Sagittal view with mask overlay
        axes[0].imshow(img_data[:, mid_slice, :], cmap='gray', aspect='auto')
        axes[0].imshow(mask_data[:, mid_slice, :], cmap='Greens', alpha=0.3, aspect='auto')
        axes[0].set_title(f'Middle Sagittal Slice with Mask')
        axes[0].axis('off')
        
        # Coronal view with mask overlay
        axes[1].imshow(img_data[mid_slice, :, :], cmap='gray', aspect='auto')
        axes[1].imshow(mask_data[mid_slice, :, :], cmap='Greens', alpha=0.3, aspect='auto')
        axes[1].set_title(f'Middle Coronal Slice with Mask')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.savefig(slice_path)
        plt.close()
        
        logger.info(f"Saved slice visualization with mask to: {slice_path}")
        
    except Exception as e:
        logger.error(f"Error generating visualization for {image_path}: {str(e)}")

def visualize_registration_results(fixed_path, moving_path, registered_path, mask_path, output_dir, subject_id):
    """Create additional visualization for registration results with mask overlay."""
    try:
        # Load images and mask
        fixed = sitk.ReadImage(fixed_path)
        moving = sitk.ReadImage(moving_path)
        registered = sitk.ReadImage(registered_path)
        mask = sitk.ReadImage(mask_path)
        
        # Convert to numpy arrays
        fixed_array = sitk.GetArrayFromImage(fixed)
        moving_array = sitk.GetArrayFromImage(moving)
        registered_array = sitk.GetArrayFromImage(registered)
        mask_array = sitk.GetArrayFromImage(mask)
        
        # Get middle slice for each image - make sure it's within bounds
        mid_slice_fixed = min(fixed_array.shape[2] // 2, fixed_array.shape[2] - 1)
        mid_slice_moving = min(moving_array.shape[2] // 2, moving_array.shape[2] - 1)
        mid_slice_registered = min(registered_array.shape[2] // 2, registered_array.shape[2] - 1)
        mid_slice_mask = min(mask_array.shape[2] // 2, mask_array.shape[2] - 1)
        
        # Create overlay visualization
        overlay_path = os.path.join(output_dir, f'{subject_id}_registration_overlay.png')
        
        # Create figure with subplots
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # Fixed image with mask
        axes[0, 0].imshow(fixed_array[:, mid_slice_fixed, :], cmap='gray')
        axes[0, 0].imshow(mask_array[:, mid_slice_mask, :], cmap='Greens', alpha=0.3)
        axes[0, 0].set_title('Fixed Image (FLAIR) - Sagittal')
        axes[0, 0].axis('off')
        
        axes[1, 0].imshow(fixed_array[mid_slice_fixed, :, :], cmap='gray')
        axes[1, 0].imshow(mask_array[mid_slice_mask, :, :], cmap='Greens', alpha=0.3)
        axes[1, 0].set_title('Fixed Image (FLAIR) - Coronal')
        axes[1, 0].axis('off')
        
        # Moving image with mask
        axes[0, 1].imshow(moving_array[:, mid_slice_moving, :], cmap='gray')
        axes[0, 1].imshow(mask_array[:, mid_slice_mask, :], cmap='Greens', alpha=0.3)
        axes[0, 1].set_title('Moving Image - Sagittal')
        axes[0, 1].axis('off')
        
        axes[1, 1].imshow(moving_array[mid_slice_moving, :, :], cmap='gray')
        axes[1, 1].imshow(mask_array[mid_slice_mask, :, :], cmap='Greens', alpha=0.3)
        axes[1, 1].set_title('Moving Image - Coronal')
        axes[1, 1].axis('off')
        
        # Registered image with mask
        axes[0, 2].imshow(registered_array[:, mid_slice_registered, :], cmap='gray')
        axes[0, 2].imshow(mask_array[:, mid_slice_mask, :], cmap='Greens', alpha=0.3)
        axes[0, 2].set_title('Registered Image - Sagittal')
        axes[0, 2].axis('off')
        
        axes[1, 2].imshow(registered_array[mid_slice_registered, :, :], cmap='gray')
        axes[1, 2].imshow(mask_array[mid_slice_mask, :, :], cmap='Greens', alpha=0.3)
        axes[1, 2].set_title('Registered Image - Coronal')
        axes[1, 2].axis('off')
        
        plt.tight_layout()
        plt.savefig(overlay_path)
        plt.close()
        
        logger.info(f"Saved registration overlay visualization with mask to: {overlay_path}")
        
    except Exception as e:
        logger.error(f"Error generating registration visualization: {str(e)}")

# Main preprocessing loop
def run_preprocessing(data_root, preprocessed_data_root, mask_root_for_raw):
    """Main preprocessing pipeline"""
    try:
        # Create necessary directories
        os.makedirs(preprocessed_data_root, exist_ok=True)
        os.makedirs(os.path.join(preprocessed_data_root, 'visualizations'), exist_ok=True)
        
        # Get all subject directories
        all_raw_subject_dirs = sorted(glob.glob(os.path.join(data_root, 'sub-*')))
        
        # Process each subject
        for raw_subject_dir in tqdm(all_raw_subject_dirs, desc="Preprocessing"):
            try:
                raw_subject_id = os.path.basename(raw_subject_dir)
                output_subject_dir = os.path.join(preprocessed_data_root, raw_subject_id, 'ses-0001')
                os.makedirs(output_subject_dir, exist_ok=True)
                
                # Get input file paths
                current_raw_dwi_path = os.path.join(raw_subject_dir, 'ses-0001', 'dwi', f'{raw_subject_id}_ses-0001_dwi.nii.gz')
                current_raw_adc_path = os.path.join(raw_subject_dir, 'ses-0001', 'dwi', f'{raw_subject_id}_ses-0001_adc.nii.gz')
                current_raw_flair_path = os.path.join(raw_subject_dir, 'ses-0001', 'anat', f'{raw_subject_id}_ses-0001_FLAIR.nii.gz')
                current_raw_mask_path = os.path.join(mask_root_for_raw, raw_subject_id, 'ses-0001', f'{raw_subject_id}_ses-0001_msk.nii.gz')
                
                # Verify files exist
                if not all(os.path.exists(f) for f in [current_raw_dwi_path, current_raw_adc_path, current_raw_flair_path, current_raw_mask_path]):
                    logger.warning(f"Skipping {raw_subject_id}: Missing required files")
                    continue
                
                # Step 1: N4 Bias Correction
                logger.info(f"Processing {raw_subject_id}: N4 Bias Correction")
                n4_dwi_path = os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_n4.nii.gz')
                n4_adc_path = os.path.join(output_subject_dir, f'{raw_subject_id}_adc_n4.nii.gz')
                n4_flair_path = os.path.join(output_subject_dir, f'{raw_subject_id}_flair_n4.nii.gz')
                
                apply_n4_bias_correction(current_raw_dwi_path, n4_dwi_path)
                apply_n4_bias_correction(current_raw_adc_path, n4_adc_path)
                apply_n4_bias_correction(current_raw_flair_path, n4_flair_path)
                
                # Step 2: Nyul Normalization (Optional)
                logger.info(f"Processing {raw_subject_id}: Nyul Normalization")
                try:
                    # Load pre-trained scales (you need to train these first)
                    dwi_standard_scale = np.load(os.path.join(preprocessed_data_root, 'dwi_nyul_scale.npy'))
                    adc_standard_scale = np.load(os.path.join(preprocessed_data_root, 'adc_nyul_scale.npy'))
                    flair_standard_scale = np.load(os.path.join(preprocessed_data_root, 'flair_nyul_scale.npy'))
                    
                    nyul_dwi_path = os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_n4_nyul.nii.gz')
                    nyul_adc_path = os.path.join(output_subject_dir, f'{raw_subject_id}_adc_n4_nyul.nii.gz')
                    nyul_flair_path = os.path.join(output_subject_dir, f'{raw_subject_id}_flair_n4_nyul.nii.gz')
                    
                    apply_nyul_normalization(n4_dwi_path, nyul_dwi_path, dwi_standard_scale)
                    apply_nyul_normalization(n4_adc_path, nyul_adc_path, adc_standard_scale)
                    apply_nyul_normalization(n4_flair_path, nyul_flair_path, flair_standard_scale)
                    
                except FileNotFoundError:
                    logger.warning(f"Skipping Nyul normalization for {raw_subject_id}: Standard scales not found")
                    nyul_dwi_path = n4_dwi_path
                    nyul_adc_path = n4_adc_path
                    nyul_flair_path = n4_flair_path
                
                # Step 3: Registration
                logger.info(f"Processing {raw_subject_id}: Registration")
                fixed_image_for_reg = nyul_flair_path
                moving_dwi_for_reg = nyul_dwi_path
                moving_adc_for_reg = nyul_adc_path
                
                # Register DWI
                register_image_and_mask(fixed_image_for_reg, moving_dwi_for_reg, current_raw_mask_path, 
                                      output_subject_dir, raw_subject_id, 'dwi')
                
                # Register ADC
                register_image_and_mask(fixed_image_for_reg, moving_adc_for_reg, current_raw_mask_path, 
                                      output_subject_dir, raw_subject_id, 'adc')
                
                # Step 4: Save FLAIR as reference
                logger.info(f"Processing {raw_subject_id}: Saving reference")
                shutil.copy(fixed_image_for_reg, os.path.join(output_subject_dir, f'{raw_subject_id}_flair_preprocessed.nii.gz'))
                
                # Step 5: Generate Visualizations
                logger.info(f"Processing {raw_subject_id}: Generating visualizations")
                plot_intensity_histogram(nyul_dwi_path, os.path.join(output_subject_dir, 'dwi_histogram.png'))
                plot_intensity_histogram(nyul_adc_path, os.path.join(output_subject_dir, 'adc_histogram.png'))
                plot_intensity_histogram(nyul_flair_path, os.path.join(output_subject_dir, 'flair_histogram.png'))
                
                # Step 5: Generate Visualizations
                logger.info(f"Processing {raw_subject_id}: Generating visualizations")
                create_comprehensive_overlay(
                    nyul_flair_path,
                    nyul_dwi_path,  # or nyul_adc_path depending on which modality you're processing
                    os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_registered.nii.gz'),  # or _adc_registered
                    current_raw_mask_path,
                    output_subject_dir,
                    raw_subject_id
                )
                # Create registration visualization
                visualize_registration_results(
                    nyul_flair_path,
                    nyul_dwi_path,
                    os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_registered.nii.gz'),
                    output_subject_dir,
                    raw_subject_id
                )
                
            except Exception as e:
                logger.error(f"Error processing subject {raw_subject_id}: {str(e)}")
                continue
                
    except Exception as e:
        logger.error(f"Error in main preprocessing loop: {str(e)}")
        raise

# Example usage
# Update these paths with your actual data directories
base_project_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
base_data_dir = os.path.join(base_project_dir, 'data')

data_root =base_data_dir
mask_root_for_raw = os.path.join(base_data_dir, 'derivatives')

data_root = "../data"
preprocessed_data_root="../data\ISLES-2022_preprocessed"

# Run the preprocessing
#run_preprocessing(data_root, preprocessed_data_root, mask_root_for_raw)

In [ ]:
import os
import shutil
import numpy as np
import SimpleITK as sitk
from tqdm import tqdm
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def apply_n4_bias_correction(input_path, output_path):
    input_image = sitk.ReadImage(input_path)
    # Convert to float32
    input_image = sitk.Cast(input_image, sitk.sitkFloat32)
    
    # Try to get mask path
    mask_path = input_path.replace('.nii.gz', '_msk.nii.gz')
    if os.path.exists(mask_path):
        mask_image = sitk.ReadImage(mask_path)
    else:
        mask_image = None
    
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    number_fitting_levels = 4
    
    # If no mask is available, just use the input image
    output_image = corrector.Execute(input_image, mask_image) if mask_image else corrector.Execute(input_image)
    
    # Convert back to original pixel type if needed
    output_image = sitk.Cast(output_image, sitk.sitkInt16)
    
    sitk.WriteImage(output_image, output_path)

def register_image_and_mask(fixed_image_path, moving_image_path, mask_path, output_dir, subject_id, modality):
    # Read images
    fixed_image = sitk.ReadImage(fixed_image_path)
    moving_image = sitk.ReadImage(moving_image_path)
    
    # Get original spacing
    fixed_spacing = fixed_image.GetSpacing()
    moving_spacing = moving_image.GetSpacing()
    
    # Fix spacing issues
    def fix_spacing(spacing):
        spacing = tuple(1.0 if x == 0 else x for x in spacing)
        spacing = tuple(abs(x) for x in spacing)
        spacing = tuple(max(x, 0.1) for x in spacing)
        return spacing
    
    fixed_spacing = fix_spacing(fixed_spacing)
    moving_spacing = fix_spacing(moving_spacing)
    
    # Set corrected spacing
    fixed_image.SetSpacing(fixed_spacing)
    moving_image.SetSpacing(moving_spacing)
    
    # Ensure images have the same dimension
    if fixed_image.GetDimension() != moving_image.GetDimension():
        raise ValueError(f"Fixed and moving images must have the same dimension. Fixed: {fixed_image.GetDimension()}, Moving: {moving_image.GetDimension()}")
    
    # Set up registration
    registration_method = sitk.ImageRegistrationMethod()
    
    # Set the metric
    registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    
    # Set the interpolator
    registration_method.SetInterpolator(sitk.sitkLinear)
    
    # Set the optimizer
    registration_method.SetOptimizerAsGradientDescent(
        learningRate=1.0,
        numberOfIterations=100,
        estimateLearningRate=sitk.ImageRegistrationMethod.EachIteration
    )
    
    # Set initial transform
    initial_transform = sitk.CenteredTransformInitializer(
        fixed_image, moving_image,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY
    )
    registration_method.SetInitialTransform(initial_transform)
    
    # Add logging
    registration_method.AddCommand(sitk.sitkStartEvent, lambda: logger.info(f"Start Registration {modality}"))
    registration_method.AddCommand(sitk.sitkIterationEvent, lambda: logger.info(f"Iteration {registration_method.GetOptimizerIteration()}"))
    
    # Cast to float32
    fixed_image = sitk.Cast(fixed_image, sitk.sitkFloat32)
    moving_image = sitk.Cast(moving_image, sitk.sitkFloat32)
    
    try:
        final_transform = registration_method.Execute(fixed_image, moving_image)
        
        # Log registration results
        logger.info(f"Final metric value: {registration_method.GetMetricValue()}")
        logger.info(f"Optimizer's stopping condition, {registration_method.GetOptimizerStopConditionDescription()}")
        
        # Get fixed image properties
        fixed_size = fixed_image.GetSize()
        fixed_spacing = fixed_image.GetSpacing()
        fixed_origin = fixed_image.GetOrigin()
        fixed_direction = fixed_image.GetDirection()
        
        # Create resampling filter
        resample = sitk.ResampleImageFilter()
        resample.SetSize(fixed_size)
        resample.SetOutputSpacing(fixed_spacing)
        resample.SetOutputOrigin(fixed_origin)
        resample.SetOutputDirection(fixed_direction)
        resample.SetTransform(final_transform)
        resample.SetInterpolator(sitk.sitkLinear)
        
        # Resample moving image
        resampled_image = resample.Execute(moving_image)
        
        # Save the registered image
        output_image_path = os.path.join(output_dir, f'{subject_id}_{modality}_registered.nii.gz')
        sitk.WriteImage(resampled_image, output_image_path)
        
        # If mask exists, apply the same transform to it
        if os.path.exists(mask_path):
            mask_image = sitk.ReadImage(mask_path)
            
            # Create mask resampling filter
            mask_resample = sitk.ResampleImageFilter()
            mask_resample.SetSize(fixed_size)
            mask_resample.SetOutputSpacing(fixed_spacing)
            mask_resample.SetOutputOrigin(fixed_origin)
            mask_resample.SetOutputDirection(fixed_direction)
            mask_resample.SetTransform(final_transform)
            mask_resample.SetInterpolator(sitk.sitkNearestNeighbor)
            
            resampled_mask = mask_resample.Execute(mask_image)
            sitk.WriteImage(resampled_mask, os.path.join(output_dir, f'{subject_id}_{modality}_mask_registered.nii.gz'))
        
        return output_image_path
        
    except Exception as e:
        logger.error(f"Registration failed for {modality}: {str(e)}")
        logger.error(f"Fixed image size: {fixed_image.GetSize()}")
        logger.error(f"Moving image size: {moving_image.GetSize()}")
        logger.error(f"Fixed image spacing: {fixed_image.GetSpacing()}")
        logger.error(f"Moving image spacing: {moving_image.GetSpacing()}")
        raise
    
def create_comprehensive_overlay(fixed_image_path, moving_image_path, registered_moving_path, mask_path, output_dir, subject_id):
    """
    Create a comprehensive overlay visualization.
    """
    try:
        logger.info(f"Creating overlay for {subject_id}")
        
        # Read images
        fixed_image = sitk.ReadImage(fixed_image_path)
        moving_image = sitk.ReadImage(moving_image_path)
        registered_moving = sitk.ReadImage(registered_moving_path)
        mask_image = sitk.ReadImage(mask_path)
        
        # Convert to numpy arrays
        fixed_array = sitk.GetArrayFromImage(fixed_image)
        moving_array = sitk.GetArrayFromImage(moving_image)
        registered_array = sitk.GetArrayFromImage(registered_moving)
        mask_array = sitk.GetArrayFromImage(mask_image)
        
        # Resample mask to match fixed image dimensions
        mask_resampled = sitk.Resample(mask_image, fixed_image, 
                                      sitk.Transform(), 
                                      sitk.sitkNearestNeighbor, 
                                      0.0, mask_image.GetPixelID())
        mask_resampled_array = sitk.GetArrayFromImage(mask_resampled)
        
        # Create RGB visualization
        # Normalize images to 0-255 range
        fixed_array_norm = ((fixed_array - fixed_array.min()) / 
                           (fixed_array.max() - fixed_array.min()) * 255).astype(np.uint8)
        registered_array_norm = ((registered_array - registered_array.min()) / 
                               (registered_array.max() - registered_array.min()) * 255).astype(np.uint8)
        
        # Create RGB image
        # FLAIR in red channel
        # DWI in green channel
        # Mask in blue channel
        rgb_image = np.zeros((fixed_array.shape[0], fixed_array.shape[1], fixed_array.shape[2], 3), dtype=np.uint8)
        
        # Add FLAIR (red)
        rgb_image[..., 0] = fixed_array_norm
        
        # Add DWI (green)
        rgb_image[..., 1] = registered_array_norm
        
        # Add mask (blue)
        rgb_image[..., 2][mask_resampled_array > 0] = 255
        
        # Save RGB visualization
        rgb_image_sitk = sitk.GetImageFromArray(rgb_image)
        rgb_image_sitk.CopyInformation(fixed_image)
        sitk.WriteImage(rgb_image_sitk, os.path.join(output_dir, f'{subject_id}_flair_dwi_overlay.nii.gz'))
        
        logger.info(f"RGB overlay created for {subject_id}")
        
        # Create separate visualization files for each component
        # Save normalized FLAIR
        fixed_image_norm = sitk.GetImageFromArray(fixed_array_norm)
        fixed_image_norm.CopyInformation(fixed_image)
        sitk.WriteImage(fixed_image_norm, os.path.join(output_dir, f'{subject_id}_flair_normalized.nii.gz'))
        
        # Save normalized DWI
        registered_image_norm = sitk.GetImageFromArray(registered_array_norm)
        registered_image_norm.CopyInformation(fixed_image)
        sitk.WriteImage(registered_image_norm, os.path.join(output_dir, f'{subject_id}_dwi_normalized.nii.gz'))
        
        # Save mask overlay
        mask_overlay = np.zeros_like(fixed_array, dtype=np.uint8)
        mask_overlay[mask_resampled_array > 0] = 255
        mask_overlay_sitk = sitk.GetImageFromArray(mask_overlay)
        mask_overlay_sitk.CopyInformation(fixed_image)
        sitk.WriteImage(mask_overlay_sitk, os.path.join(output_dir, f'{subject_id}_mask_overlay.nii.gz'))
        
        logger.info(f"Individual overlays created for {subject_id}")
        
    except Exception as e:
        logger.error(f"Error creating overlay for {subject_id}: {str(e)}")
        
def preprocess_subject(subject_id, raw_data_root, mask_root_for_raw, preprocessed_data_root):
    try:
        logger.info(f"Processing {subject_id}")
        
        # Create output directory using absolute path
        output_subject_dir = os.path.abspath(os.path.join(preprocessed_data_root, subject_id, 'ses-0001'))
        os.makedirs(output_subject_dir, exist_ok=True)
        
        # Get input paths using absolute paths
        current_raw_dwi_path = os.path.abspath(os.path.join(raw_data_root, subject_id, 'ses-0001', 'dwi', f'{subject_id}_ses-0001_dwi.nii.gz'))
        current_raw_adc_path = os.path.abspath(os.path.join(raw_data_root, subject_id, 'ses-0001', 'dwi', f'{subject_id}_ses-0001_adc.nii.gz'))
        current_raw_flair_path = os.path.abspath(os.path.join(raw_data_root, subject_id, 'ses-0001', 'anat', f'{subject_id}_ses-0001_FLAIR.nii.gz'))
        
        # Use the correct mask file naming convention
        current_raw_mask_path = os.path.abspath(os.path.join(mask_root_for_raw, subject_id, 'ses-0001', f'{subject_id}_ses-0001_msk.nii.gz'))
        
        # Log paths for debugging
        logger.info(f"Output directory: {output_subject_dir}")
        logger.info(f"DWI path: {current_raw_dwi_path}")
        logger.info(f"ADC path: {current_raw_adc_path}")
        logger.info(f"FLAIR path: {current_raw_flair_path}")
        logger.info(f"Mask path: {current_raw_mask_path}")
        
        # Check if files exist
        if not all(os.path.exists(f) for f in [current_raw_dwi_path, current_raw_adc_path, current_raw_flair_path, current_raw_mask_path]):
            logger.warning(f"Skipping {subject_id}: Missing required files")
            logger.warning(f"  DWI: {os.path.exists(current_raw_dwi_path)}")
            logger.warning(f"  ADC: {os.path.exists(current_raw_adc_path)}")
            logger.warning(f"  FLAIR: {os.path.exists(current_raw_flair_path)}")
            logger.warning(f"  Mask: {os.path.exists(current_raw_mask_path)}")
            logger.warning(f"  Mask path: {current_raw_mask_path}")
            return
            
        # Apply N4 bias correction
        logger.info(f"Processing {subject_id}: N4 Bias Correction")
        
        # Create output paths using absolute paths
        n4_dwi_path = os.path.abspath(os.path.join(output_subject_dir, f'{subject_id}_dwi_n4.nii.gz'))
        n4_adc_path = os.path.abspath(os.path.join(output_subject_dir, f'{subject_id}_adc_n4.nii.gz'))
        n4_flair_path = os.path.abspath(os.path.join(output_subject_dir, f'{subject_id}_flair_n4.nii.gz'))
        
        # Log output paths for debugging
        logger.info(f"N4 DWI output: {n4_dwi_path}")
        logger.info(f"N4 ADC output: {n4_adc_path}")
        logger.info(f"N4 FLAIR output: {n4_flair_path}")
        
        apply_n4_bias_correction(current_raw_dwi_path, n4_dwi_path)
        apply_n4_bias_correction(current_raw_adc_path, n4_adc_path)
        apply_n4_bias_correction(current_raw_flair_path, n4_flair_path)
        
        # Copy mask file
        shutil.copy(current_raw_mask_path, os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'))
        
        # Registration
        logger.info(f"Processing {subject_id}: Registration")
        fixed_image_for_reg = n4_flair_path
        
        # Register FLAIR itself
        register_image_and_mask(
            fixed_image_for_reg,
            n4_flair_path,
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id,
            'flair'
        )
        
        # Register DWI
        register_image_and_mask(
            fixed_image_for_reg,
            n4_dwi_path,
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id,
            'dwi'
        )
        # Register DWI
        register_image_and_mask(
            fixed_image_for_reg,
            n4_dwi_path,
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id,
            'dwi'
        )
        
        # Register ADC
        register_image_and_mask(
            fixed_image_for_reg,
            n4_adc_path,
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id,
            'adc'
        )
        
        # Generate visualizations
        logger.info(f"Processing {subject_id}: Generating visualizations")
        create_comprehensive_overlay(
            n4_flair_path,
            n4_dwi_path,
            os.path.join(output_subject_dir, f'{subject_id}_dwi_registered.nii.gz'),
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id
        )
        
    except Exception as e:
        logger.error(f"Error processing subject {subject_id}: {str(e)}")
        raise
        logger.error(f"Error processing subject {subject_id}: {str(e)}")
        raise

def main_preprocessing():
    # Define absolute paths
    base_data_dir = os.path.abspath("../data")
    raw_data_root = base_data_dir
    mask_root_for_raw = os.path.join(base_data_dir, 'derivatives')
    preprocessed_data_root = os.path.abspath("../data/ISLES-2022_preprocessed")
    
    # Create preprocessed root directory
    os.makedirs(preprocessed_data_root, exist_ok=True)
    
    # Get list of subjects
    subjects = [d for d in os.listdir(raw_data_root) if d.startswith('sub-')]
    
    # Process each subject
    for subject_id in tqdm(subjects, desc="Preprocessing"):
        preprocess_subject(subject_id, raw_data_root, mask_root_for_raw, preprocessed_data_root)

if __name__ == "__main__":
    main_preprocessing()

Preprocessing:   0%|          | 0/199 [00:00<?, ?it/s]INFO:__main__:Processing sub-strokecase0051
INFO:__main__:Output directory: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\ISLES-2022_preprocessed\sub-strokecase0051\ses-0001
INFO:__main__:DWI path: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\sub-strokecase0051\ses-0001\dwi\sub-strokecase0051_ses-0001_dwi.nii.gz
INFO:__main__:ADC path: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\sub-strokecase0051\ses-0001\dwi\sub-strokecase0051_ses-0001_adc.nii.gz
INFO:__main__:FLAIR path: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\sub-strokecase0051\ses-0001\anat\sub-strokecase0051_ses-0001_FLAIR.nii.gz
INFO:__main__:Mask path: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\derivatives\sub-strokecase0051\ses-0001\sub-strokecase0051_ses-0001_msk.nii.gz
INFO:__main__:Processing sub-strok

TypeError: in method 'ResampleImageFilter_SetOutputOrigin', argument 2 of type 'std::vector< double,std::allocator< double > >'

In [21]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm, Act

class MultiEncoderUNet(nn.Module):
    def __init__(self, in_channels_dwi, in_channels_adc, in_channels_flair, out_channels,
                 spatial_dims=3, channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2)):
        super().__init__()
        # Separate encoders for each modality [2, 32, 33, 46]
        self.encoder_dwi = UNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels_dwi,
            out_channels=out_channels, # This output is just before bottleneck
            channels=channels,
            strides=strides,
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU
        ).encoder # Access the encoder part

        self.encoder_adc = UNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels_adc,
            out_channels=out_channels,
            channels=channels,
            strides=strides,
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU
        ).encoder

        self.encoder_flair = UNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels_flair,
            out_channels=out_channels,
            channels=channels,
            strides=strides,
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU
        ).encoder

        # Shared decoder (output channels for each encoder are summed at bottleneck)
        # The input channels to the decoder will be sum of bottleneck channels from all encoders
        # For simplicity, let's assume the last channel in 'channels' is the bottleneck feature size
        bottleneck_features = channels[-1] * 3 # Assuming 3 modalities
        self.decoder = UNet(
            spatial_dims=spatial_dims,
            in_channels=bottleneck_features,
            out_channels=out_channels,
            channels=channels[::-1], # Reverse channels for decoder
            strides=strides[::-1], # Reverse strides for decoder
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU,
            is_decoder=True # Indicate this is the decoder part
        ) # This is a simplified representation. A true MultiEncoderUNet would manage skip connections carefully.

    def forward(self, x):
        # x is expected to be a dictionary or a concatenated tensor
        # For this simplified example, assume x is already concatenated
        # In a real MONAI pipeline, you'd pass a dictionary and handle it with transforms
        # For demonstration, let's assume input x has 3 channels for DWI, ADC, FLAIR
        dwi_input = x[:, 0:1, :, :, :] # Assuming channel 0 is DWI
        adc_input = x[:, 1:2, :, :, :] # Assuming channel 1 is ADC
        flair_input = x[:, 2:3, :, :, :] # Assuming channel 2 is FLAIR

        # Encode each modality
        dwi_features = self.encoder_dwi(dwi_input)
        adc_features = self.encoder_adc(adc_input)
        flair_features = self.encoder_flair(flair_input)

        # Concatenate features at bottleneck (simplified, actual nnU-Net handles skip connections)
        # This is a conceptual bottleneck fusion. Actual Multi-encoder nnU-Net combines features
        # from corresponding encoder layers to feed into the decoder's skip connections.
        # For this simplified UNet decoder, we'll just concatenate the deepest features.
        fused_bottleneck = torch.cat([dwi_features, adc_features, flair_features], dim=1) #  for deepest features

        # Pass through shared decoder
        # This is a placeholder. A true UNet decoder requires skip connections from encoders.
        # For a full implementation, consider adapting MONAI's UNet or SegResNet to accept
        # multiple encoder outputs and merge them into the decoder's skip pathways.
        # For now, we'll pass the fused bottleneck into a simple decoder.
        # A more accurate implementation would require custom UNet structure or MONAI's support for multi-input.
        # For this example, let's use a standard UNet and assume the input has 3 channels (DWI, ADC, FLAIR)
        # and the UNet handles the multi-channel input directly as a single input.
        # The "Multi-encoder" part would be handled by the data pipeline preparing the input.
        # The previous section's `ConcatItemsd` already creates a 3-channel input.
        # So, we revert to a standard UNet for simplicity of demonstration,
        # but note that Multi-encoder nnU-Net is conceptually superior.
        pass

# Revert to standard MONAI UNet for demonstration, assuming concatenated input
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = UNet(
    spatial_dims=3,
    in_channels=3, # 3 modalities: DWI, ADC, FLAIR
    out_channels=1, # Binary segmentation: lesion/non-lesion
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH,
    act=Act.LEAKYRELU
).to(device)

# For a true Multi-encoder nnU-Net, one would build a custom network that takes
# a dictionary of images (e.g., {'dwi': tensor, 'adc': tensor, 'flair': tensor})
# and processes them through separate encoders before combining features.
# This is more complex than a simple UNet and would require a custom MONAI network definition.

c:\Users\punit\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
INFO:transformers.file_utils:TensorFlow version 2.13.0 available.
INFO:transformers.file_utils:PyTorch version 2.7.1+cpu available.
INFO:transformers.modeling_xlnet:Better speed can be achieved with apex installed from https://www.github.com/nvidia/apex .


NameError: name 'nn' is not defined

In [4]:
loss_function = DiceCELoss(to_onehot_y=False, sigmoid=True) # Combines Dice Loss and Cross-Entropy Loss [35, 36, 44]
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
dice_metric = DiceMetric(include_background=False, reduction="mean") # For evaluation [40]

In [29]:
# Import MONAI and other required packages
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ResizeWithPadOrCropd,
    ScaleIntensityRanged,
    NormalizeIntensityd,
    RandSpatialCropd,
    RandFlipd,
    RandGaussianNoised,
    RandAdjustContrastd,
    RandBiasFieldd,
    ToTensord,
    ConcatItemsd
)
from monai.data import (
    Dataset,
    CacheDataset,
    DataLoader
)
import torch
import numpy as np
from pathlib import Path

# Define data loading function
def load_preprocessed_data(preprocessed_root):
    data_list = []
    for subject_dir in Path(preprocessed_root).glob("sub-*"):
        subject_id = subject_dir.name
        data_list.append({
            "dwi": str(subject_dir / "ses-0001" / f"{subject_id}_dwi_n4.nii.gz"),
            "adc": str(subject_dir / "ses-0001" / f"{subject_id}_adc_n4.nii.gz"),
            "flair": str(subject_dir / "ses-0001" / f"{subject_id}_flair_n4.nii.gz"),
            "label": str(subject_dir / "ses-0001" / f"{subject_id}_lesion-msk.nii.gz"),
            "subject_id": subject_id
        })
    return data_list

# Load preprocessed data
preprocessed_root = "../data/ISLES-2022_preprocessed"
preprocessed_data = load_preprocessed_data(preprocessed_root)

# Define transforms for each modality
modality_transforms = Compose([
    LoadImaged(keys=["dwi", "adc", "flair", "label"]),
    EnsureChannelFirstd(keys=["dwi", "adc", "flair", "label"]),
    Orientationd(keys=["dwi", "adc", "flair", "label"], axcodes="RAS"),
    Spacingd(
        keys=["dwi", "adc", "flair", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=["bilinear", "bilinear", "bilinear", "nearest"],
        padding_mode=["zeros", "zeros", "zeros", "zeros"],
        align_corners=[True, True, True, True]
    ),
    ResizeWithPadOrCropd(
        keys=["dwi", "adc", "flair", "label"],
        spatial_size=(224, 224, 224),  # Use a standard size that works for most brains
        mode=["constant", "constant", "constant", "constant"]
    ),
    ScaleIntensityRanged(keys=["dwi", "adc", "flair"], a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
    NormalizeIntensityd(keys=["dwi", "adc", "flair"], subtrahend=0.5, divisor=0.5),
    ConcatItemsd(keys=["dwi", "adc", "flair"], name="image"),
    ToTensord(keys=["image", "label"])
])

# Create dataset
dataset = CacheDataset(
    data=preprocessed_data,
    transform=modality_transforms,
    cache_rate=1.0,
    num_workers=4
)

# Create data loaders
train_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available()
)

Loading dataset: 100%|██████████| 5/5 [00:07<00:00,  1.56s/it]


In [52]:
# Import MONAI and other required packages
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ResizeWithPadOrCropd,
    ScaleIntensityRanged,
    NormalizeIntensityd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandRotate90d,
    RandGaussianNoised,
    RandScaleIntensityd,
    RandShiftIntensityd,
    ConcatItemsd,
    ToTensord
)
from monai.data import (
    Dataset,
    CacheDataset,
    DataLoader,
    list_data_collate
)
from sklearn.model_selection import train_test_split
import torch
import numpy as np
from pathlib import Path
import random

# Define data loading function
def load_preprocessed_data(preprocessed_root):
    data_list = []
    for subject_dir in Path(preprocessed_root).glob("sub-*"):
        subject_id = subject_dir.name
        data_list.append({
            "dwi": str(subject_dir / "ses-0001" / f"{subject_id}_dwi_n4.nii.gz"),
            "adc": str(subject_dir / "ses-0001" / f"{subject_id}_adc_n4.nii.gz"),
            "flair": str(subject_dir / "ses-0001" / f"{subject_id}_flair_n4.nii.gz"),
            "label": str(subject_dir / "ses-0001" / f"{subject_id}_lesion-msk.nii.gz"),
            "subject_id": subject_id
        })
    return data_list

# Load preprocessed data
preprocessed_root = "../data/ISLES-2022_preprocessed"
preprocessed_data = load_preprocessed_data(preprocessed_root)

# Split data into train and validation sets
train_data, val_data = train_test_split(preprocessed_data, test_size=0.2, random_state=42)

# Enhanced training transforms
train_transforms = Compose([
    LoadImaged(keys=["dwi", "adc", "flair", "label"]),
    EnsureChannelFirstd(keys=["dwi", "adc", "flair", "label"]),
    Orientationd(keys=["dwi", "adc", "flair", "label"], axcodes="RAS"),
    Spacingd(
        keys=["dwi", "adc", "flair", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=["bilinear", "bilinear", "bilinear", "nearest"],
        padding_mode=["zeros", "zeros", "zeros", "zeros"],
        align_corners=[True, True, True, True]
    ),
    ResizeWithPadOrCropd(
        keys=["dwi", "adc", "flair", "label"],
        spatial_size=(224, 224, 224),
        mode=["constant", "constant", "constant", "constant"]
    ),
    ScaleIntensityRanged(keys=["dwi", "adc", "flair"], a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
    NormalizeIntensityd(keys=["dwi", "adc", "flair"], subtrahend=0.5, divisor=0.5),
    
    # Advanced augmentations
    RandCropByPosNegLabeld(
        keys=["dwi", "adc", "flair", "label"],
        label_key="label",
        spatial_size=(128, 128, 128),
        pos=3,
        neg=1,
        num_samples=4
    ),
    
    # Basic spatial augmentations
    RandFlipd(keys=["dwi", "adc", "flair", "label"], spatial_axis=0, prob=0.5),
    RandFlipd(keys=["dwi", "adc", "flair", "label"], spatial_axis=1, prob=0.5),
    RandFlipd(keys=["dwi", "adc", "flair", "label"], spatial_axis=2, prob=0.5),
    RandRotate90d(keys=["dwi", "adc", "flair", "label"], prob=0.5, spatial_axes=(0, 1)),
    RandRotate90d(keys=["dwi", "adc", "flair", "label"], prob=0.5, spatial_axes=(1, 2)),
    RandRotate90d(keys=["dwi", "adc", "flair", "label"], prob=0.5, spatial_axes=(0, 2)),
    
    # Intensity augmentations
    RandGaussianNoised(keys=["dwi", "adc", "flair"], prob=0.1, mean=0.0, std=0.01),
    RandScaleIntensityd(keys=["dwi", "adc", "flair"], factors=0.1, prob=0.1),
    RandShiftIntensityd(keys=["dwi", "adc", "flair"], offsets=0.1, prob=0.1),
    
    ConcatItemsd(keys=["dwi", "adc", "flair"], name="image"),
    ToTensord(keys=["image", "label"])
])

# Create datasets
train_dataset = CacheDataset(
    data=train_data,
    transform=train_transforms,
    cache_rate=1.0,
    num_workers=4
)

val_dataset = CacheDataset(
    data=val_data,
    transform=train_transforms,
    cache_rate=1.0,
    num_workers=4
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate
)

# Define the model
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
import torch.optim as optim

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
model = UNet(
    spatial_dims=3,
    in_channels=3,  # DWI, ADC, FLAIR
    out_channels=2,  # Background and lesion
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2
).to(device)

# Loss function
loss_function = DiceCELoss(to_onehot_y=True, sigmoid=True, squared_pred=True)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Dice metric
dice_metric = DiceMetric(include_background=False, reduction="mean")

# Training parameters
num_epochs = 100
val_interval = 5
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

# Training loop
# Post-processing transforms
post_pred = Compose([
    Activations(sigmoid=True),
    AsDiscrete(threshold_values=True)
])

# For labels, we need to ensure they're in the correct format
post_label = Compose([
    AsDiscrete(to_onehot=2)  # Convert to one-hot with 2 classes
])

# Update the validation loop
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        print(f"{step}/{len(train_loader)}, train_loss: {loss.item():.4f}")
        
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"Epoch {epoch + 1} average loss: {epoch_loss:.4f}")
    
    # Validation
    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                
                # Sliding window inference
                roi_size = (128, 128, 128)
                sw_batch_size = 4
                val_outputs = sliding_window_inference(val_inputs, roi_size, sw_batch_size, model)
                
                # Post-process predictions and labels
                val_outputs = post_pred(val_outputs)
                val_labels = post_label(val_labels)
                
                # Update dice metric
                dice_metric(y_pred=val_outputs, y=val_labels)
                
            # Calculate and print metrics
            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            metric_values.append(metric)
            
            # Update learning rate
            scheduler.step(metric)
            
            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), "best_metric_model.pth")
                print("saved new best metric model")
                
            print(f"Current epoch: {epoch + 1} current mean dice: {metric:.4f}")
            print(f"Best mean dice: {best_metric:.4f} at epoch: {best_metric_epoch}")

Loading dataset: 100%|██████████| 1/1 [00:01<00:00,  2.00s/it]


Epoch 1/100
1/2, train_loss: 1.3235
2/2, train_loss: 1.2974
Epoch 1 average loss: 1.3105
Epoch 2/100
1/2, train_loss: 1.2854
2/2, train_loss: 1.2617
Epoch 2 average loss: 1.2735
Epoch 3/100
1/2, train_loss: 1.2486
2/2, train_loss: 1.2520
Epoch 3 average loss: 1.2503
Epoch 4/100
1/2, train_loss: 1.2287
2/2, train_loss: 1.2321
Epoch 4 average loss: 1.2304
Epoch 5/100
1/2, train_loss: 1.2199
2/2, train_loss: 1.2072
Epoch 5 average loss: 1.2136


RuntimeError: applying transform <monai.transforms.post.array.AsDiscrete object at 0x000001A65EF13890>

In [ ]:
# Add these imports at the top with other imports
from monai.transforms import AsDiscrete
from monai.data import decollate_batch
import matplotlib.pyplot as plt
import os

metric = dice_metric.aggregate().item()
dice_metric.reset()
metric_values.append(metric)
            
if metric > best_metric:
    best_metric = metric
    best_metric_epoch = epoch + 1
    torch.save(model.state_dict(), "best_metric_model.pth")
     print("saved new best metric model")
                
            print(f"Current epoch: {epoch + 1} current mean dice: {metric:.4f}")
            print(f"Best mean dice: {best_metric:.4f} at epoch: {best_metric_epoch}")

# Visualization function
def visualize_results(image, label, pred, subject_id, slice_idx=112):
    plt.figure("check", (18, 6))
    
    plt.subplot(1, 3, 1)
    plt.title("image (DWI)")
    plt.imshow(image[0, 0, :, :, slice_idx], cmap="gray")
    
    plt.subplot(1, 3, 2)
    plt.title("label")
    plt.imshow(label[0, 0, :, :, slice_idx])
    
    plt.subplot(1, 3, 3)
    plt.title("prediction")
    plt.imshow(pred[0, 0, :, :, slice_idx])
    
    plt.savefig(f"results/{subject_id}_results.png")
    plt.close()

# Post-processing function
def postprocess_prediction(prediction):
    # Apply thresholding
    prediction = (prediction > 0.5).float()
    
    # Remove small connected components
    prediction = prediction.cpu().numpy()
    prediction = remove_small_objects(prediction, min_size=100)
    
    # Fill holes
    prediction = binary_fill_holes(prediction)
    
    return torch.from_numpy(prediction).float().to(prediction.device)

# Add this after the training loop
# Create results directory
os.makedirs("results", exist_ok=True)

# Test the model with the best weights
model.load_state_dict(torch.load("best_metric_model.pth"))
model.eval()

with torch.no_grad():
    for val_data in val_loader:
        val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
        roi_size = (128, 128, 128)
        sw_batch_size = 4
        
        # Sliding window inference
        val_outputs = sliding_window_inference(val_inputs, roi_size, sw_batch_size, model)
        val_outputs = torch.sigmoid(val_outputs)
        
        # Post-process predictions
        val_outputs = postprocess_prediction(val_outputs)
        
        # Calculate dice score
        dice_score = dice_metric(y_pred=val_outputs, y=val_labels)
        print(f"Subject {val_data['subject_id'][0]} Dice score: {dice_score.item():.4f}")
        
        # Save visualization
        visualize_results(
            val_inputs.cpu().numpy(),
            val_labels.cpu().numpy(),
            val_outputs.cpu().numpy(),
            val_data["subject_id"][0]
        )

IndentationError: unexpected indent (4008419763.py, line 8)